# Validation + calibration analysis (for report sections 5.2 and 5.3)

Regenerates the held-out metrics and figures for the v7 model.

**Why this is fast:** the holdout was defined by clustering the *provided*
coordinates only (seed 42), so it reproduces without touching the external data.
Only ~3.9k images need encoding, not 269k. Runtime ~5 minutes.

**Attach:** the competition dataset + the `Geogs_v7` notebook output.
**Settings:** GPU T4, Internet OFF.

Outputs 3 PNGs to `/kaggle/working/` plus a printed table you can paste into LaTeX.

In [1]:
# =====================================================================
# CELL 1 - Setup
# =====================================================================
import os
os.environ["HF_HUB_OFFLINE"]="1"; os.environ["TRANSFORMERS_OFFLINE"]="1"
import gc, json, glob, time, random, warnings
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, torch, torch.nn as nn
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
from torch.utils.data import Dataset, DataLoader
from PIL import Image
from sklearn.cluster import MiniBatchKMeans
from transformers import CLIPVisionModel, AutoModel

T0=time.time()
def log(m): print(f"[+{(time.time()-T0)/60:5.1f}m] {m}", flush=True)
SEED=42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
torch.backends.cudnn.deterministic=True; torch.backends.cudnn.benchmark=False
R=6371.0088; IMG,NV,BS=224,2,64
DEV=torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
DEV1=torch.device("cuda:1") if torch.cuda.device_count()>=2 else DEV

def find_all(p,r="/kaggle/input"): return sorted(glob.glob(os.path.join(r,"**",p),recursive=True))
def pick(cols,*k):
    for c in cols:
        lc=c.lower().replace("_","").replace(" ","")
        if all(x in lc for x in k): return c
def l2v(lat,lon):
    la,lo=np.radians(np.asarray(lat,float)),np.radians(np.asarray(lon,float))
    return np.stack([np.cos(la)*np.cos(lo),np.cos(la)*np.sin(lo),np.sin(la)],-1)
def v2ll(v):
    v=np.asarray(v,float); v=v/(np.linalg.norm(v,axis=-1,keepdims=True)+1e-12)
    return np.degrees(np.arcsin(np.clip(v[...,2],-1,1))), np.degrees(np.arctan2(v[...,1],v[...,0]))
def hav(a,b,c,d):
    a,b,c,d=map(lambda x:np.radians(np.asarray(x,float)),(a,b,c,d))
    h=np.sin((c-a)/2)**2+np.cos(a)*np.cos(c)*np.sin((d-b)/2)**2
    return 2*R*np.arcsin(np.sqrt(np.clip(h,0,1)))
log("ready")

[+  0.0m] ready


In [2]:
# =====================================================================
# CELL 2 - Load v7 artifacts
# =====================================================================
cands=sorted(set(os.path.dirname(p) for p in find_all("calibration.json")))
v7=[d for d in cands if "v7" in d.lower()]
ART=(v7 or cands)[0]
log(f"artifacts: {ART}")
HEADS=sorted(glob.glob(os.path.join(ART,"head_seed*.pt"))) or sorted(glob.glob(os.path.join(ART,"head_fold*.pt")))
MU=np.load(f"{ART}/feat_mu.npy"); SD=np.load(f"{ART}/feat_sd.npy")
CF=np.load(f"{ART}/fine_centroids.npy"); CELLC=np.load(f"{ART}/cell_country.npy")
CAL=json.load(open(f"{ART}/calibration.json"))
LAM=float(CAL.get("lam",0.0)); ALPHA=float(CAL.get("alpha",4.2))
FLOOR=float(CAL.get("floor",15.0)); RMAX=float(CAL.get("r_max",3000.0)); TOPK=int(CAL.get("topk",8))
log(f"calibration: {CAL}")

sd0=torch.load(HEADS[0],map_location="cpu")
FEAT=sd0["trunk.0.weight"].shape[1]; HID=sd0["trunk.0.weight"].shape[0]
NF=sd0["fine.weight"].shape[0]; NC=sd0["coarse.weight"].shape[0]; NK=sd0["country.weight"].shape[0]
USE_DINO=(FEAT==4096)
log(f"feat={FEAT} hid={HID} fine={NF} country={NK} heads={len(HEADS)} dino={USE_DINO}")

[+  0.2m] artifacts: /kaggle/input/notebooks/mitraasrinivasan1367/geogs-v7/artifacts
[+  0.2m] calibration: {'blend': 0.0, 'lam': 0.0, 'alpha': 4.200000000000001, 'floor': 15.0, 'score': 0.37716166448491106, 'w_provided': 8.0, 'use_dino': True}
[+  0.2m] feat=4096 hid=1024 fine=2000 country=298 heads=5 dino=True


In [3]:
# =====================================================================
# CELL 3 - Rebuild the EXACT holdout split (provided data only, seed 42)
# =====================================================================
GT=[p for p in find_all("*.csv")+find_all("*.xlsx")
    if "ground" in os.path.basename(p).lower() or "coordinate" in os.path.basename(p).lower()][0]
gt=pd.read_excel(GT) if GT.endswith(".xlsx") else pd.read_csv(GT)
GLAT=pick(gt.columns,"lat"); GLON=pick(gt.columns,"lon") or pick(gt.columns,"lng")
GID=pick(gt.columns,"id") or gt.columns[0]
log(f"ground truth: {GT} | {len(gt)} rows")

train_imgs=[]
for d in glob.glob("/kaggle/input/**/training_dataset",recursive=True):
    for e in ("*.jpg","*.jpeg","*.png"):
        train_imgs+=glob.glob(os.path.join(d,"**",e),recursive=True)
by={os.path.splitext(os.path.basename(p))[0]:p for p in sorted(set(train_imgs))}
log(f"training images on disk: {len(by)}")

gt["stem"]=gt[GID].astype(str).map(lambda x: os.path.splitext(str(x))[0])
gt["path"]=gt["stem"].map(by.get)
df=pd.DataFrame({"path":gt["path"].values,
                 "lat":pd.to_numeric(gt[GLAT],errors="coerce").values,
                 "lon":pd.to_numeric(gt[GLON],errors="coerce").values})
df=df[df.path.notna()].dropna(subset=["lat","lon"])
df=df[df.lat.between(-90,90)&df.lon.between(-180,180)].reset_index(drop=True)
log(f"provided rows: {len(df)}")

V=l2v(df.lat.values,df.lon.values)
nb=min(2000,max(6,int(len(df)//12)))
kb=MiniBatchKMeans(nb,random_state=SEED,batch_size=4096,n_init=5).fit(V)
perm=np.random.RandomState(SEED).permutation(nb)
fold_of={b:i%5 for i,b in enumerate(perm)}
is_hold=np.array([fold_of[b]==0 for b in kb.labels_])
HOLD=np.where(is_hold)[0]
log(f"blocks={nb} | HOLDOUT={len(HOLD)} images")
log(">>> v7's log recorded 3908. If this matches, the split reproduced exactly.")

[+  0.3m] ground truth: /kaggle/input/datasets/mitraasrinivasan/geo-guessr-final-hackathon-evaluation/geo-guessr-final-hackathon-evaluation/training_dataset/noised_dataset/ground_truth_coordinates.csv | 19002 rows
[+  0.3m] training images on disk: 19002
[+  0.3m] provided rows: 19002
[+  0.5m] blocks=1583 | HOLDOUT=3908 images
[+  0.5m] >>> v7's log recorded 3908. If this matches, the split reproduced exactly.


In [4]:
# =====================================================================
# CELL 4 - Encode holdout images
# =====================================================================
CLIP_MEAN=[0.48145466,0.4578275,0.40821073]; CLIP_STD=[0.26862954,0.26130258,0.27577711]
DINO_MEAN=[0.485,0.456,0.406];                  DINO_STD=[0.229,0.224,0.225]
def load_bb(kind,dev):
    if kind=="clip":
        m=CLIPVisionModel.from_pretrained(f"{ART}/clip_vit_l14").to(dev).half().eval()
        mu,st=CLIP_MEAN,CLIP_STD
    else:
        m=AutoModel.from_pretrained(f"{ART}/dinov2_large").to(dev).half().eval()
        mu,st=DINO_MEAN,DINO_STD
    for p in m.parameters(): p.requires_grad=False
    return dict(model=m,kind=kind,dim=m.config.hidden_size,dev=dev,
                mean=torch.tensor(mu,dtype=torch.float32,device=dev).view(1,3,1,1).half(),
                std =torch.tensor(st,dtype=torch.float32,device=dev).view(1,3,1,1).half())

class ViewDS(Dataset):
    def __init__(s,p): s.p=list(p)
    def __len__(s): return len(s.p)
    def __getitem__(s,i):
        try: im=Image.open(s.p[i]).convert("RGB")
        except Exception: im=Image.new("RGB",(IMG,IMG),(128,128,128))
        w,h=im.size; q=min(w,h); l,t=(w-q)//2,(h-q)//2
        a=im.resize((IMG,IMG),Image.BICUBIC)
        b=im.crop((l,t,l+q,t+q)).resize((IMG,IMG),Image.BICUBIC)
        return torch.stack([torch.from_numpy(np.asarray(a,dtype=np.uint8)).permute(2,0,1),
                            torch.from_numpy(np.asarray(b,dtype=np.uint8)).permute(2,0,1)])

def fwd(bk,x,n):
    xx=x.to(bk["dev"],non_blocking=True).reshape(n*NV,3,IMG,IMG).half().div_(255.)
    o=bk["model"](pixel_values=(xx-bk["mean"])/bk["std"])
    f=o.pooler_output if bk["kind"]=="clip" else o.last_hidden_state[:,0]
    return f.reshape(n,NV*bk["dim"])

@torch.no_grad()
def encode(paths):
    bc=load_bb("clip",DEV); bd=load_bb("dino",DEV1) if USE_DINO else None
    dl=DataLoader(ViewDS(paths),batch_size=BS,shuffle=False,num_workers=4,pin_memory=True)
    A,B,seen=[],[],0
    for x in dl:
        n=x.shape[0]
        A.append(fwd(bc,x,n).float().cpu().numpy())
        if USE_DINO: B.append(fwd(bd,x,n).float().cpu().numpy())
        seen+=n
        if seen%(BS*10)<BS: log(f"  {seen}/{len(paths)}")
    bc["model"].cpu()
    if USE_DINO: bd["model"].cpu()
    gc.collect(); torch.cuda.empty_cache()
    return np.concatenate([np.concatenate(A),np.concatenate(B)],1) if USE_DINO else np.concatenate(A)

hp=df.path.values[HOLD]
log(f"encoding {len(hp)} holdout images ...")
Z=encode(hp)
Xh=torch.tensor((Z-MU)/SD,dtype=torch.float32,device=DEV)
log(f"features {Z.shape}")

[+  0.5m] encoding 3908 holdout images ...


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/439 [00:00<?, ?it/s]

[+  1.2m]   640/3908
[+  1.5m]   1280/3908
[+  1.9m]   1920/3908
[+  2.3m]   2560/3908
[+  2.7m]   3200/3908
[+  3.1m]   3840/3908
[+  3.2m] features (3908, 4096)


In [5]:
# =====================================================================
# CELL 5 - Predict with the ensemble
# =====================================================================
CFT=torch.tensor(CF,dtype=torch.float32,device=DEV)
CCT=torch.tensor(np.where(CELLC>=0,CELLC,0),device=DEV)
CCV=torch.tensor((CELLC>=0).astype(np.float32),device=DEV)

class GeoHead(nn.Module):
    def __init__(s,d,nf,nc,nk,hid):
        super().__init__()
        s.trunk=nn.Sequential(nn.Linear(d,hid),nn.LayerNorm(hid),nn.GELU(),nn.Dropout(0.30),
                              nn.Linear(hid,hid),nn.LayerNorm(hid),nn.GELU(),nn.Dropout(0.15))
        s.fine=nn.Linear(hid,nf); s.coarse=nn.Linear(hid,nc); s.country=nn.Linear(hid,nk)
        s.delta=nn.Linear(hid,3); s.unc=nn.Linear(hid,1)
    def forward(s,x):
        h=s.trunk(x); return s.fine(h),s.coarse(h),s.country(h),s.delta(h),s.unc(h).squeeze(-1)

def decode(fl,delta,topk=TOPK,ms=600.0):
    p=torch.softmax(fl.float(),1); w,idx=torch.topk(p,topk,1)
    a=CFT[idx[:,0]]; cand=CFT[idx]
    cos=(cand*a.unsqueeze(1)).sum(-1).clamp(-1+1e-9,1-1e-9)
    w=w*((R*torch.acos(cos))<=ms).float(); w=w/(w.sum(1,keepdim=True)+1e-9)
    base=(cand*w.unsqueeze(-1)).sum(1); base=base/(base.norm(dim=1,keepdim=True)+1e-9)
    v=base+0.08*torch.tanh(delta); return v/(v.norm(dim=1,keepdim=True)+1e-9)

def cadj(fl,cl,lam):
    if lam<=0: return fl
    lp=torch.log_softmax(fl.float(),1); pc=torch.softmax(cl.float(),1)
    return lp+lam*torch.log(pc[:,CCT]*CCV.unsqueeze(0)+1e-6)

Tf=torch.zeros(len(Xh),NF,device=DEV); Tc=torch.zeros(len(Xh),NK,device=DEV)
Td=torch.zeros(len(Xh),3,device=DEV);  Tu=torch.zeros(len(Xh),device=DEV)
with torch.no_grad():
    for h in HEADS:
        m=GeoHead(FEAT,NF,NC,NK,HID).to(DEV); m.load_state_dict(torch.load(h,map_location=DEV)); m.eval()
        fl,cl,ctl,dl_,ul=m(Xh)
        Tf+=torch.softmax(fl.float(),1); Tc+=torch.softmax(ctl.float(),1)
        Td+=dl_.float(); Tu+=ul.float(); del m
K=len(HEADS); Tf/=K; Tc/=K; Td/=K; Tu/=K

v=decode(cadj(torch.log(Tf+1e-12),torch.log(Tc+1e-12),LAM),Td)
plat,plon=v2ll(v.cpu().numpy())
err=hav(plat,plon,df.lat.values[HOLD],df.lon.values[HOLD])
rad=np.clip(ALPHA*np.maximum(np.expm1(Tu.cpu().numpy()),1.0),FLOOR,RMAX)
log(f"median error {np.median(err):.1f} km | median radius {np.median(rad):.0f} km")

# country accuracy
import shapely
from shapely.geometry import shape
from shapely.strtree import STRtree
geoms=[]
gj=find_all("*.geojson")
if gj:
    for f_ in json.load(open(gj[0],encoding="utf-8"))["features"]:
        try:
            g=shape(f_["geometry"]); geoms.append(g if g.is_valid else g.buffer(0))
        except Exception: pass
def assign(la,lo):
    out=np.full(len(la),-1,dtype=np.int64)
    if not geoms: return out
    t=STRtree(geoms); pts=shapely.points(np.asarray(lo,float),np.asarray(la,float))
    pr=t.query(pts,predicate="intersects"); out[pr[0]]=pr[1]; return out
true_c=assign(df.lat.values[HOLD],df.lon.values[HOLD])
pred_c=assign(plat,plon)
in_country=float(((pred_c==true_c)&(true_c>=0)).mean())
head_acc=float((Tc.argmax(1).cpu().numpy()==true_c)[true_c>=0].mean())
log(f"in-country {100*in_country:.1f}% | country-head acc {100*head_acc:.1f}%")
np.save("/kaggle/working/holdout_err.npy",err); np.save("/kaggle/working/holdout_rad.npy",rad)

[+  3.2m] median error 582.8 km | median radius 801 km
[+  3.4m] in-country 50.4% | country-head acc 64.0%


In [6]:
# =====================================================================
# CELL 6 - FIGURES + TABLES for the report
# =====================================================================
plt.rcParams.update({"figure.dpi":150,"font.size":9})

# --- Fig 1: error distribution ---
fig,ax=plt.subplots(figsize=(5.2,3.2))
ax.hist(np.clip(err,1,20000),bins=np.logspace(0,4.4,50),color="#1A3C6E",alpha=.85)
ax.set_xscale("log"); ax.set_xlabel("Great-circle error (km, log scale)"); ax.set_ylabel("Held-out images")
for q,c in [(50,"#B3261E"),(25,"#888"),(75,"#888")]:
    ax.axvline(np.percentile(err,q),color=c,ls="--",lw=1)
ax.set_title(f"Held-out error distribution (n={len(err)}, median {np.median(err):.0f} km)")
fig.tight_layout(); fig.savefig("/kaggle/working/fig_error_dist.png"); plt.close(fig)

# --- Fig 2: reliability - coverage by claimed-radius decile ---
order=np.argsort(rad); bins=np.array_split(order,10)
cov=[float((err[b]<=rad[b]).mean()) for b in bins]
mid=[float(np.median(rad[b])) for b in bins]
fig,ax=plt.subplots(figsize=(5.2,3.2))
ax.plot(mid,[100*c for c in cov],"o-",color="#1A3C6E")
ax.axhline(100*float((err<=rad).mean()),color="#B3261E",ls="--",lw=1,
           label=f"overall {100*(err<=rad).mean():.1f}%")
ax.set_xscale("log"); ax.set_xlabel("Claimed radius (km, decile median)")
ax.set_ylabel("Coverage (%)"); ax.set_ylim(0,100); ax.legend()
ax.set_title("Calibration: does the true point fall inside the claimed radius?")
fig.tight_layout(); fig.savefig("/kaggle/working/fig_calibration.png"); plt.close(fig)

# --- Fig 3: claimed radius vs actual error ---
fig,ax=plt.subplots(figsize=(5.2,3.2))
ax.scatter(rad,np.clip(err,1,20000),s=3,alpha=.18,color="#1A3C6E",edgecolors="none")
lim=[max(rad.min(),10),RMAX]
ax.plot(lim,lim,"--",color="#B3261E",lw=1,label="radius = error")
ax.set_xscale("log"); ax.set_yscale("log")
ax.set_xlabel("Claimed radius (km)"); ax.set_ylabel("Actual error (km)")
ax.legend(); ax.set_title("Points below the line are covered by their radius")
fig.tight_layout(); fig.savefig("/kaggle/working/fig_radius_vs_error.png"); plt.close(fig)
log("saved 3 PNGs to /kaggle/working/")

# --- LaTeX tables ---
print("\n===== 5.2 error percentiles =====")
print("\\begin{tabular}{@{}lr@{}}\\toprule")
print("Statistic & Value \\\\ \\midrule")
for q in [10,25,50,75,90]:
    print(f"{q}th percentile error & {np.percentile(err,q):,.0f} km \\\\")
print(f"Mean error & {err.mean():,.0f} km \\\\")
print(f"Country-head accuracy & {100*head_acc:.1f}\\% \\\\")
print(f"Predicted point in correct country & {100*in_country:.1f}\\% \\\\")
print(f"Held-out images & {len(err):,} \\\\")
print("\\bottomrule\\end{tabular}")

print("\n===== 5.3 calibration by decile =====")
print("\\begin{tabular}{@{}rrr@{}}\\toprule")
print("Decile & Median radius (km) & Coverage \\\\ \\midrule")
for i,(m,c) in enumerate(zip(mid,cov)):
    print(f"{i+1} & {m:,.0f} & {100*c:.1f}\\% \\\\")
print("\\midrule")
print(f"Overall & {np.median(rad):,.0f} & {100*(err<=rad).mean():.1f}\\% \\\\")
print("\\bottomrule\\end{tabular}")

[+  3.4m] saved 3 PNGs to /kaggle/working/

===== 5.2 error percentiles =====
\begin{tabular}{@{}lr@{}}\toprule
Statistic & Value \\ \midrule
10th percentile error & 138 km \\
25th percentile error & 271 km \\
50th percentile error & 583 km \\
75th percentile error & 2,121 km \\
90th percentile error & 8,762 km \\
Mean error & 2,408 km \\
Country-head accuracy & 64.0\% \\
Predicted point in correct country & 50.4\% \\
Held-out images & 3,908 \\
\bottomrule\end{tabular}

===== 5.3 calibration by decile =====
\begin{tabular}{@{}rrr@{}}\toprule
Decile & Median radius (km) & Coverage \\ \midrule
1 & 478 & 64.7\% \\
2 & 571 & 60.9\% \\
3 & 647 & 67.0\% \\
4 & 706 & 65.0\% \\
5 & 770 & 61.6\% \\
6 & 834 & 58.1\% \\
7 & 906 & 50.4\% \\
8 & 993 & 58.6\% \\
9 & 1,134 & 51.5\% \\
10 & 1,454 & 46.4\% \\
\midrule
Overall & 801 & 58.4\% \\
\bottomrule\end{tabular}
